# E046 — Controlled Best-Generator Dataset

Cette campagne génère un **nouveau dataset propre**. Elle ne recycle pas les
306 372 anciens PNG génériques ignorés par E045.

## Règles scientifiques

- vérité QR logicielle principale : **qr-scanner-wechat via qr-verify@0.2.0** ;
- payload exact uniquement ;
- 37 presets, trois répétitions conservatrices ;
- OpenCV, ZBar et ZXing ne votent pas dans le score principal ;
- raster brut toujours conservé ;
- aucune bordure blanche/uniforme éligible comme sortie finale ;
- variante `scene_preserving` : pas de crop et cœur 580×580 octet-identique ;
- Stage 1, Stage 2, latent et tous les checkpoints SR-MPGD sont persistés ;
- le téléphone reste la vérité finale et n'est pas simulé comme acquis.

Le notebook fonctionne également en mode partiel, uniquement lorsqu'aucun Job
GPU E046 n'est actif :

```powershell
.\scripts\e046-remote.ps1 -Partial
```

Le script refuse de démarrer Jupyter pendant une génération afin de ne pas
concurrencer la RTX.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display, Markdown, Image

OUTPUT_ROOT = Path(os.environ.get(
    "PROOFTAG_E046_OUTPUT_ROOT",
    "/data/e046-controlled-best-generator-v1",
))
latest = json.loads((OUTPUT_ROOT / "LATEST.json").read_text(encoding="utf-8"))
R = Path(latest["plan_dir"])
plan = json.loads((R / "plan.json").read_text(encoding="utf-8"))
verdict_path = R / "verdict.json"
complete_path = R / "COMPLETE.json"
verdict = (
    json.loads(verdict_path.read_text(encoding="utf-8"))
    if verdict_path.is_file()
    else None
)

def load_rows():
    final = R / "dataset/e046-observations.json"
    if final.is_file():
        return json.loads(final.read_text(encoding="utf-8"))

    rows = []
    for path in sorted((R / "parents").glob("*/scoring/comparison.json")):
        rows.extend(json.loads(path.read_text(encoding="utf-8")))
    for path in sorted((R / "refinements").glob("*/*/scoring/comparison.json")):
        rows.extend(json.loads(path.read_text(encoding="utf-8")))
    return rows

rows = load_rows()
df = pd.DataFrame(rows)
for column, default in (
    ("srmpgd_recipe_id", None),
    ("row_id", None),
    ("pixel_duplicate", False),
    ("projection_was_active", False),
):
    if column not in df.columns:
        df[column] = default
print("Plan E046 :", R)
print("Profile   :", plan["profile"])
print("Status    :", latest.get("status"))
print("Rows      :", len(df))
print("Engine QR :", plan["qr_software_engine"])

## 1. État d'avancement et contrat

In [ ]:
parent_total = len(plan["candidates"])
parent_generated = sum(
    (R / "parents" / item["id"] / "GENERATION_COMPLETE.json").is_file()
    for item in plan["candidates"]
)
parent_scored = sum(
    (R / "parents" / item["id"] / "SCORING_COMPLETE.json").is_file()
    for item in plan["candidates"]
)
selected_path = R / "selected-parents.json"
selected = (
    json.loads(selected_path.read_text(encoding="utf-8"))
    if selected_path.is_file()
    else {"selected": []}
)
refinement_tasks = [
    (item["candidate_id"], recipe["id"])
    for item in selected["selected"]
    for recipe in plan["srmpgd_recipes"]
]
refinement_generated = sum(
    (R / "refinements" / candidate / recipe / "GENERATION_COMPLETE.json").is_file()
    for candidate, recipe in refinement_tasks
)
refinement_scored = sum(
    (R / "refinements" / candidate / recipe / "SCORING_COMPLETE.json").is_file()
    for candidate, recipe in refinement_tasks
)

state = pd.DataFrame([
    ["Parents prévus", parent_total],
    ["Parents générés", parent_generated],
    ["Parents scorés", parent_scored],
    ["Parents sélectionnés", len(selected["selected"])],
    ["Trajectoires SR-MPGD prévues", len(refinement_tasks)],
    ["Trajectoires générées", refinement_generated],
    ["Trajectoires scorées", refinement_scored],
    ["Agrégation complète", complete_path.is_file()],
], columns=["Étape", "Valeur"])
display(state)

In [ ]:
display({
    "source_commit": plan["source_commit"],
    "E045_plan_id": plan["e045_plan_id"],
    "E045_manifest_sha256": plan["e045_manifest_sha256"],
    "scientific_plan_hash": plan["scientific_plan_hash"],
    "primary_label": plan["qr_primary_label"],
    "other_decoders": plan["other_decoders_role"],
    "production_ready": False if verdict is None else verdict["production_ready"],
})

## 2. Prompts, familles, payloads et compositions

In [ ]:
candidate_df = pd.DataFrame(plan["candidates"])
display(candidate_df[[
    "id", "prompt_id", "prompt_family", "prompt_variant_index",
    "parent_recipe_id", "seed", "payload", "quiet_zone_hint", "prompt"
]].style.set_properties(subset=["prompt", "quiet_zone_hint"], **{"white-space": "normal"}))

In [ ]:
family_counts = candidate_df["prompt_family"].value_counts()
plt.figure(figsize=(11, 4))
plt.bar(family_counts.index, family_counts.values)
plt.ylabel("Candidats")
plt.xlabel("Famille visuelle")
plt.title("Couverture des familles E046")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 3. Recettes Stage 1 / Stage 2

In [ ]:
parent_recipes = pd.DataFrame(plan["parent_recipes"])
display(parent_recipes.style.set_properties(
    subset=["rationale"], **{"white-space": "normal"}
))

In [ ]:
coverage = parent_recipes[[
    "qr_mask_pattern", "error_correction", "stage1_steps",
    "stage1_guidance_scale", "stage1_controlnet_scale",
    "stage2_initialization", "stage2_strength", "stage2_steps",
    "stage2_controlnet_scale", "stage2_qr_weight",
    "stage2_perceptual_weight",
]]
display(coverage)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
scatter = ax.scatter(
    parent_recipes["stage2_qr_weight"],
    parent_recipes["stage2_perceptual_weight"],
    s=90,
)
for _, row in parent_recipes.iterrows():
    ax.annotate(
        f"m{row['qr_mask_pattern']}",
        (row["stage2_qr_weight"], row["stage2_perceptual_weight"]),
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set_xscale("log")
ax.set_xlabel("Poids SRG Stage 2")
ax.set_ylabel("Poids perceptuel Stage 2")
ax.set_title("Espace initial Stage 2, masques 0 à 7")
plt.tight_layout()
plt.show()

## 4. Recettes SR-MPGD — jamais forcées sur tous les parents

In [ ]:
srmpgd_recipes = pd.DataFrame(plan["srmpgd_recipes"])
display(srmpgd_recipes.style.set_properties(
    subset=["rationale"], **{"white-space": "normal"}
))

In [ ]:
if not srmpgd_recipes.empty:
    plt.figure(figsize=(9, 5))
    plt.scatter(
        srmpgd_recipes["gamma"],
        srmpgd_recipes["latent_radius_rms"],
        s=srmpgd_recipes["max_iterations"] * 20,
    )
    for _, row in srmpgd_recipes.iterrows():
        plt.annotate(row["id"], (row["gamma"], row["latent_radius_rms"]),
                     xytext=(5, 5), textcoords="offset points")
    plt.xscale("log")
    plt.xlabel("Gamma brut")
    plt.ylabel("Rayon latent RMS")
    plt.title("Gamma, trust region et nombre d'itérations")
    plt.tight_layout()
    plt.show()

## 5. Dataset disponible

In [ ]:
if df.empty:
    display(Markdown("**Aucun scoring disponible pour le moment.**"))
else:
    display(df.head(30))
    print("Colonnes :", len(df.columns))
    print("Rasters uniques :", df["image_sha256"].nunique())

## 6. Distribution WeChat exacte / 37

In [ ]:
if not df.empty:
    exact = pd.to_numeric(df["wechat_exact_presets"], errors="coerce").dropna()
    plt.figure(figsize=(10, 5))
    plt.hist(exact, bins=np.arange(-0.5, 38.5, 1))
    plt.xlabel("Presets exacts qr-scanner-wechat / 37")
    plt.ylabel("Images")
    plt.title("Distribution de la cible logicielle principale")
    plt.tight_layout()
    plt.show()

    buckets = pd.cut(
        exact,
        bins=[-1, 5, 15, 25, 35, 37],
        labels=["0–5", "6–15", "16–25", "26–35", "36–37"],
    ).value_counts().sort_index()
    display(buckets.to_frame("images"))

> Un score `22/37` signifie que 22 transformations logicielles ont rendu le
> payload exact à `qr-scanner-wechat`. Ce n'est pas un taux de réussite téléphone.

## 7. WeChat par prompt, masque, ECC et recette

In [ ]:
if not df.empty:
    best_prompt = (
        df.groupby(["prompt_id", "prompt_family"], dropna=False)["wechat_exact_presets"]
        .max()
        .sort_values(ascending=False)
        .reset_index()
    )
    display(best_prompt)

    plt.figure(figsize=(12, 5))
    plt.bar(best_prompt["prompt_id"], best_prompt["wechat_exact_presets"])
    plt.ylabel("Meilleur WeChat exact / 37")
    plt.xlabel("Prompt")
    plt.title("Sensibilité au prompt")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty:
    by_mask = (
        df.groupby("qr_mask_pattern")["wechat_exact_presets"]
        .agg(["count", "mean", "max", "median"])
        .reset_index()
    )
    display(by_mask)
    plt.figure(figsize=(9, 4))
    plt.bar(by_mask["qr_mask_pattern"].astype(str), by_mask["max"])
    plt.xlabel("Masque QR")
    plt.ylabel("Maximum exact / 37")
    plt.title("Couverture des huit masques légaux")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty:
    by_recipe = (
        df.groupby(["parent_recipe_id", "source_kind"])["wechat_exact_presets"]
        .agg(["count", "mean", "max"])
        .sort_values("max", ascending=False)
    )
    display(by_recipe)

## 8. Compromis QR / esthétique

In [ ]:
if not df.empty:
    plot = df.copy()
    for column in ("wechat_exact_presets", "clip_aesthetic", "hpsv2_1",
                   "clip_score", "lpips", "module_error_rate"):
        plot[column] = pd.to_numeric(plot[column], errors="coerce")

    paired = plot.dropna(subset=["wechat_exact_presets", "clip_aesthetic"])
    plt.figure(figsize=(10, 7))
    for source, group in paired.groupby("source_kind"):
        plt.scatter(
            group["clip_aesthetic"],
            group["wechat_exact_presets"],
            alpha=0.55,
            label=source,
        )
    plt.xlabel("CLIP-Aesthetic")
    plt.ylabel("WeChat exact / 37")
    plt.title("Frontière scannabilité logicielle / esthétique")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty:
    paired = plot.dropna(subset=["wechat_exact_presets", "hpsv2_1"])
    plt.figure(figsize=(10, 7))
    plt.scatter(
        paired["hpsv2_1"],
        paired["wechat_exact_presets"],
        alpha=0.55,
    )
    plt.xlabel("HPSv2")
    plt.ylabel("WeChat exact / 37")
    plt.title("WeChat exact vs préférence visuelle HPS")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty:
    paired = plot.dropna(subset=["module_error_rate", "wechat_exact_presets"])
    plt.figure(figsize=(10, 7))
    plt.scatter(
        paired["module_error_rate"],
        paired["wechat_exact_presets"],
        alpha=0.5,
    )
    plt.xlabel("Module error rate")
    plt.ylabel("WeChat exact / 37")
    plt.title("MER reste un diagnostic, pas la cible")
    plt.tight_layout()
    plt.show()

## 9. Quiet zone : brut contre scene-preserving

In [ ]:
if not df.empty:
    qz = df[df["quiet_zone_variant"].isin(["raw", "scene_preserving"])].copy()
    keys = [
        "candidate_id", "source_kind", "srmpgd_recipe_id", "iteration"
    ]
    paired_qz = qz.pivot_table(
        index=keys,
        columns="quiet_zone_variant",
        values=["wechat_exact_presets", "clip_aesthetic", "hpsv2_1"],
        aggfunc="first",
    )
    display(paired_qz.head(100))

In [ ]:
if not df.empty:
    qz_delta_rows = []
    for _, group in qz.groupby(keys, dropna=False):
        if set(group["quiet_zone_variant"]) != {"raw", "scene_preserving"}:
            continue
        raw = group[group["quiet_zone_variant"] == "raw"].iloc[0]
        scene = group[group["quiet_zone_variant"] == "scene_preserving"].iloc[0]
        qz_delta_rows.append({
            "candidate_id": raw["candidate_id"],
            "source_kind": raw["source_kind"],
            "srmpgd_recipe_id": raw.get("srmpgd_recipe_id"),
            "iteration": raw["iteration"],
            "delta_wechat": scene["wechat_exact_presets"] - raw["wechat_exact_presets"],
            "delta_clip_aesthetic": scene["clip_aesthetic"] - raw["clip_aesthetic"],
            "delta_hpsv2": (
                scene["hpsv2_1"] - raw["hpsv2_1"]
                if pd.notna(scene["hpsv2_1"]) and pd.notna(raw["hpsv2_1"])
                else np.nan
            ),
            "core_same": bool(scene["core_byte_identical_to_raw"]),
            "qz_guard": scene["quiet_zone_delivery_guard_pass"],
        })
    qz_delta = pd.DataFrame(qz_delta_rows)
    display(qz_delta)
    if not qz_delta.empty:
        plt.figure(figsize=(9, 5))
        plt.hist(qz_delta["delta_wechat"], bins=np.arange(-37.5, 38.5, 1))
        plt.xlabel("Gain scene-preserving – brut, presets exacts")
        plt.ylabel("Paires")
        plt.title("Effet logiciel de la composition périphérique")
        plt.tight_layout()
        plt.show()

La variante `scene_preserving` ne colle pas un cadre uniforme. Elle part de
l'œuvre, conserve les couleurs à basse fréquence, lisse les détails locaux et
éclaircit sans crop. Le cœur QR doit garder exactement le même hash.

## 10. SR-MPGD : trajectoires, gamma, projection et no-op

In [ ]:
if not df.empty:
    sr = plot[plot["source_kind"] == "srmpgd"].copy()
    if sr.empty:
        display(Markdown("Aucune trajectoire SR-MPGD scorée."))
    else:
        raw_sr = sr[sr["quiet_zone_variant"] == "raw"]
        grouped = raw_sr.groupby(
            ["candidate_id", "srmpgd_recipe_id", "gamma", "iteration"],
            dropna=False,
        )["wechat_exact_presets"].max().reset_index()
        for (candidate, recipe), group in grouped.groupby(
            ["candidate_id", "srmpgd_recipe_id"]
        ):
            plt.figure(figsize=(8, 4))
            plt.plot(group["iteration"], group["wechat_exact_presets"], marker="o")
            plt.xlabel("Itération")
            plt.ylabel("WeChat exact / 37")
            plt.title(f"{candidate}\n{recipe}")
            plt.ylim(-0.5, 37.5)
            plt.tight_layout()
            plt.show()

In [ ]:
if not df.empty and "projection_was_active" in df:
    projection = (
        df[df["source_kind"] == "srmpgd"]
        .groupby(["srmpgd_recipe_id", "gamma"], dropna=False)
        .agg(
            rows=("variant", "count"),
            projection_active=("projection_was_active", "sum"),
            max_wechat=("wechat_exact_presets", "max"),
        )
    )
    display(projection)

In [ ]:
if not df.empty and "pixel_duplicate" in df:
    duplicate = (
        df.groupby(["source_kind", "stage"])["pixel_duplicate"]
        .agg(["count", "sum"])
    )
    display(duplicate)

## 11. Gardes visuelles et erreurs techniques

In [ ]:
if not df.empty:
    guard = (
        df.groupby(["source_kind", "quiet_zone_variant"])["visual_guard_pass"]
        .agg(["count", "sum", "mean"])
    )
    display(guard)

In [ ]:
failure_paths = sorted((R / "failures").glob("*.json"))
print("Fichiers d'échec :", len(failure_paths))
for path in failure_paths[:50]:
    display(json.loads(path.read_text(encoding="utf-8")))

## 12. Pareto et gagnants

In [ ]:
pareto_path = R / "dataset/pareto-front.json"
if pareto_path.is_file():
    pareto_df = pd.DataFrame(json.loads(pareto_path.read_text(encoding="utf-8")))
    pareto_columns = [
        "candidate_id", "source_kind", "srmpgd_recipe_id", "variant",
        "iteration", "gamma", "wechat_exact_presets",
        "wechat_original_exact", "clip_aesthetic", "hpsv2_1",
        "clip_score", "lpips", "image_path"
    ]
    display(pareto_df.reindex(columns=pareto_columns))
else:
    display(Markdown("Pareto disponible après agrégation."))

In [ ]:
best_path = R / "dataset/best-by-prompt.json"
if best_path.is_file():
    best_df = pd.DataFrame(json.loads(best_path.read_text(encoding="utf-8")))
    best_columns = [
        "prompt_id", "candidate_id", "source_kind", "variant",
        "srmpgd_recipe_id", "iteration", "wechat_exact_presets",
        "clip_aesthetic", "hpsv2_1", "image_path"
    ]
    display(best_df.reindex(columns=best_columns))

## 13. Galerie des meilleurs candidats

In [ ]:
def show_gallery(rows, title, columns=4, limit=24):
    rows = list(rows)[:limit]
    if not rows:
        display(Markdown(f"**{title} : aucune image.**"))
        return
    nrows = math.ceil(len(rows) / columns)
    fig, axes = plt.subplots(nrows, columns, figsize=(16, 4.5 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row in zip(axes, rows):
        path = Path(str(row["image_path"]))
        if path.is_file():
            axis.imshow(PILImage.open(path).convert("RGB"))
        axis.set_title(
            f"{row['prompt_id']}\n{row['source_kind']} {row['variant']}\n"
            f"WeChat {row['wechat_exact_presets']}/37",
            fontsize=9,
        )
        axis.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

if best_path.is_file():
    show_gallery(
        json.loads(best_path.read_text(encoding="utf-8")),
        "Meilleur candidat par prompt",
    )
elif not df.empty:
    fallback = (
        df[df["eligible_final"] == True]
        .sort_values(["wechat_exact_presets", "clip_aesthetic"], ascending=False)
        .drop_duplicates("prompt_id")
        .to_dict("records")
    )
    show_gallery(fallback, "Meilleurs candidats partiels")

## 14. Pipeline complète du gagnant

In [ ]:
if verdict is None:
    display(Markdown("Verdict final indisponible en mode partiel."))
else:
    display(verdict)
    winner_id = verdict["winner_candidate_id"]
    parent_root = R / "parents" / winner_id / "images"
    panel = [
        ("Stage 1 brut", parent_root / "stage1-raw.png"),
        ("Stage 1 scene-qz", parent_root / "stage1-scene-qz.png"),
        ("Stage 2 brut", parent_root / "stage2-raw.png"),
        ("Stage 2 scene-qz", parent_root / "stage2-scene-qz.png"),
        ("Gagnant final", R / "pipeline/99-FINAL-QR.png"),
    ]
    fig, axes = plt.subplots(1, len(panel), figsize=(22, 5))
    for axis, (label, path) in zip(axes, panel):
        if path.is_file():
            axis.imshow(PILImage.open(path).convert("RGB"))
        axis.set_title(label)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
for name in (
    "best-by-prompt-contact-sheet.png",
    "pareto-contact-sheet.png",
    "phone-sample-contact-sheet.png",
):
    path = R / "pipeline" / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))

## 15. Préparation E047 et téléphone

In [ ]:
if verdict is not None:
    readiness = {
        "software_dataset_complete": verdict["software_dataset_complete"],
        "software_advisor_training_candidate": verdict[
            "software_advisor_training_candidate"
        ],
        "automatic_advisor_training_authorized": verdict[
            "automatic_advisor_training_authorized"
        ],
        "phone_truth_available": verdict["phone_truth_available"],
        "phone_surrogate_training_authorized": verdict[
            "phone_surrogate_training_authorized"
        ],
        "production_ready": verdict["production_ready"],
        "next_action": verdict["next_action"],
    }
    display(readiness)

### Décision attendue après revue

E047 pourra apprendre les paramètres qui maximisent **WeChat exact / 37** sous
gardes esthétiques. L'entraînement automatique reste bloqué tant que :

- les splits prompt/payload/pixels ne sont pas gelés ;
- les erreurs et no-op ne sont pas audités ;
- un échantillon représentatif n'est pas sélectionné pour le téléphone.

Le surrogate téléphone et la production restent interdits sans labels physiques.